# Day 5: Pandas Data Cleaning

Load and inspect the Day 5 CSV datasets with pandas. The source files are read only and are not modified by this notebook.

In [1]:
!pip install pandas

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Support opening Jupyter from either the repository root or the day5 folder.
candidate_directories = (Path.cwd(), Path.cwd() / "day5")
data_dir = next(
    (directory for directory in candidate_directories
     if (directory / "employee_data.csv").exists()
     and (directory / "surat_house_price.csv").exists()),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Could not find employee_data.csv and surat_house_price.csv. "
        "Start Jupyter from the project root or the day5 folder."
    )

print(f"Using datasets from: {data_dir.resolve()}")

Using datasets from: C:\Users\Administrator\Downloads\spark\day5


## Load the datasets

In [3]:
employees_df = pd.read_csv(data_dir / "employee_data.csv", low_memory=False)
houses_df = pd.read_csv(data_dir / "surat_house_price.csv", low_memory=False)

datasets = {
    "employees": employees_df,
    "houses": houses_df,
}

for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape[0]:,} rows x {dataframe.shape[1]} columns")

employees: 202 rows x 9 columns
houses: 4,525 rows x 11 columns


## Preview the data

In [4]:
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} DATASET")
    display(dataframe.head())


EMPLOYEES DATASET


,EmployeeID,Name,Age,Salary,Department,JoiningDate,Contact,Experience,PerformanceScore
0,1001,James Brown,22.0,45000.0,HR,2022-03-15,james.brownyahoo.com | 6613186091,11,7.1
1,1002,Lucas Green,26.0,42000.0,Finance,2020-06-30,lucas.green@gmail.com | 8219935181,10,8.0
2,1003,Mia Scott,28.0,85000.0,NaN,2022-02-10,mia.scott@yahoo.com | 3194875749,2,NaN
3,1004,Noah Davis,25.0,85000.0,Operations,2014-11-18,noah.davis@gmail.com | 9555979711,8,7.5
4,1005,Alice Martin,NaN,85000.0,Marketing,2022-08-27,alice.martinhotmail.com | 7529170342,7,9.2



HOUSES DATASET


,property_name,areaWithType,square_feet,transaction,status,floor,furnishing,facing,description,price_per_sqft,price
0,2 BHK Apartment for Sale in Dindoli Surat,Carpet Area,644 sqft,New Property,Poss. by Oct '24,5 out of 10,Unfurnished,West,"Luxury project with basement parking, Solar rooftop, Gym, Build with advanced machinery and stan...","₹2,891 per sqft",₹33.8 Lac
1,2 BHK Apartment for Sale in Althan Surat,Super Area,1278 sqft,New Property,Poss. by Jan '26,6 out of 14,Unfurnished,South -West,2 And 3 BHK Luxurious Flat for Sell In New Althan Area With All Aminities .Read more,"₹3,551 per sqft",₹45.4 Lac
2,2 BHK Apartment for Sale in Pal Gam Surat,Super Area,1173 sqft,Resale,Ready to Move,5 out of 13,Semi-Furnished,East,"This affordable 2 BHK flat is situated along a 200foot road, ensuring easy access and connectivi...","₹3,800 per sqft",₹44.6 Lac
3,2 BHK Apartment for Sale in Jahangirabad Surat,Carpet Area,700 sqft,New Property,Ready to Move,6 out of 14,Unfurnished,East,2 BHK Flat For sell IN Jahangirabad Prime Location,"₹3,966 per sqft",₹47 Lac
4,"2 BHK Apartment for Sale in Orchid Fantasia, Palanpur Surat",Super Area,1250 sqft,Orchid Fantasia,New Property,Unfurnished,2,2,"Multistorey Apartment for Sale in Palanpur, Surat. Covered area is 1250.0 Sq-ft. This property ...","₹3,600 per sqft",₹45 Lac


## Review column types and missing values

In [5]:
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} COLUMN TYPES")
    print(dataframe.dtypes.to_string())

    missing_values = dataframe.isna().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    print("\nMissing values:")
    print(missing_values.to_string() if not missing_values.empty else "None")


EMPLOYEES COLUMN TYPES
EmployeeID            int64
Name                 object
Age                 float64
Salary              float64
Department           object
JoiningDate          object
Contact              object
Experience            int64
PerformanceScore    float64

Missing values:
Department          26
Age                 22
PerformanceScore    21
JoiningDate         19
Salary              14

HOUSES COLUMN TYPES
property_name     object
areaWithType      object
square_feet       object
transaction       object
status            object
floor             object
furnishing        object
facing            object
description       object
price_per_sqft    object
price             object

Missing values:
description       1371
facing             589
price_per_sqft     368
furnishing         340
transaction        104
floor               45
status               1


## Continue your cleaning below

Both DataFrames are ready as `employees_df` and `houses_df`.

In [6]:
# Calculate each replacement from the available employee values.
employee_fill_values = {
    "Age": employees_df["Age"].mean(),
    "Salary": employees_df["Salary"].median(),
    "Department": employees_df["Department"].mode().iloc[0],
    "PerformanceScore": employees_df["PerformanceScore"].mean(),
}

employees_df = employees_df.fillna(value=employee_fill_values)

print("Replacement values used:")
display(pd.Series(employee_fill_values, name="replacement_value"))

Replacement values used:


Age                 27.694444
Salary                56000.0
Department          Marketing
PerformanceScore     8.298895
Name: replacement_value, dtype: object

## Verify the cleaned employee columns

The check below verifies the four requested columns. `JoiningDate` is reported separately because its missing values were not included in the requested replacements.

In [7]:
cleaned_columns = ["Age", "Salary", "Department", "PerformanceScore"]
remaining_cleaned_nulls = employees_df[cleaned_columns].isna().sum()

print("Missing values in the cleaned columns:")
display(remaining_cleaned_nulls)

assert remaining_cleaned_nulls.eq(0).all(), "Some cleaned columns still contain missing values."
print("Verified: no missing values remain in the four cleaned columns.")

print("\nMissing values remaining across the full employee dataset:")
display(employees_df.isna().sum()[lambda counts: counts > 0])

Missing values in the cleaned columns:


Age                 0
Salary              0
Department          0
PerformanceScore    0
dtype: int64

Verified: no missing values remain in the four cleaned columns.

Missing values remaining across the full employee dataset:


JoiningDate    19
dtype: int64

## Convert `JoiningDate` to datetime

In [ ]:
employees_df["JoiningDate"] = pd.to_datetime(
    employees_df["JoiningDate"],
    format="%Y-%m-%d",
    errors="coerce",
)

print("JoiningDate dtype:", employees_df["JoiningDate"].dtype)
display(employees_df[["EmployeeID", "Name", "JoiningDate"]].head())

## Extract the joining year, month, and day

Nullable integer columns preserve missing date components as `<NA>`.

In [ ]:
employees_df["JoiningYear"] = employees_df["JoiningDate"].dt.year.astype("Int64")
employees_df["JoiningMonth"] = employees_df["JoiningDate"].dt.month.astype("Int64")
employees_df["JoiningDay"] = employees_df["JoiningDate"].dt.day.astype("Int64")

display(
    employees_df[
        ["EmployeeID", "JoiningDate", "JoiningYear", "JoiningMonth", "JoiningDay"]
    ].head()
)

## Count employees who joined after 2019

In [ ]:
joined_after_2019 = employees_df.loc[
    employees_df["JoiningYear"].gt(2019).fillna(False)
].copy()
employees_joined_after_2019 = len(joined_after_2019)

print(f"Employees who joined after 2019: {employees_joined_after_2019}")
display(joined_after_2019[["EmployeeID", "Name", "JoiningDate", "JoiningYear"]].head())